In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install python-calamine

import pandas as pd, numpy as np
BASE  = '/content/drive/MyDrive/Palestine_Project'
RAW   = f'{BASE}/00_raw/Palestine_Project_Data.xlsx'
CLEAN = f'{BASE}/01_clean'
ML    = f'{BASE}/02_ml'

def read_sheet(name, **kw):
    return pd.read_excel(RAW, sheet_name=name, engine='calamine', **kw)

def save_clean(df, filename):
    df.to_csv(f'{CLEAN}/{filename}', index=False, encoding='utf-8')
    print(f'saved {filename}: {len(df):,} rows x {df.shape[1]} cols')

In [ ]:
il = read_sheet('3_ILO_Unemployment').drop(columns=['note_classif'])
assert len(il) == 42579 and il['indicator'].nunique() == 58

names = {'UNE_DEAP_SEX_AGE_RT':     'Unemployment rate by sex and age',
         'UNE_DEAP_SEX_AGE_EDU_RT': 'Unemployment rate by sex, age and education',
         'UNE_TUNE_SEX_AGE_NB':     'Unemployed persons by sex and age (thousands)'}
x = il[il['indicator'].isin(names)].copy()
n_raw = len(x)
x = x[x['obs_value'].notna()]

fact = pd.DataFrame({
    'year': x['time'], 'indicator_code': x['indicator'],
    'sex': x['sex'].map({'SEX_T': 'Total', 'SEX_M': 'Male', 'SEX_F': 'Female'}),
    'age_class': x['classif1'], 'edu_class': x['classif2'],
    'value': x['obs_value'], 'obs_status': x['obs_status'],
    'series_break_flag': (x['time'] >= 2024).astype(int)})
assert not fact.duplicated(['year', 'indicator_code', 'sex', 'age_class', 'edu_class']).any()
assert fact['sex'].notna().all() and 2023 not in set(fact['year'])

dim = pd.read_csv(f'{CLEAN}/dim_indicator.csv')
dim = dim[dim['source'] != 'ILO']
dim = pd.concat([dim, pd.DataFrame({'indicator_code': list(names), 'indicator_name': list(names.values()),
                                    'theme': 'Labour', 'source': 'ILO'})], ignore_index=True)

print('rows of the 3 indicators:', n_raw, '| empty values dropped:', n_raw - len(x), '| fact rows:', len(fact))
h = fact[(fact['indicator_code'] == 'UNE_DEAP_SEX_AGE_RT') & (fact['sex'] == 'Total') & (fact['age_class'] == 'AGE_YTHADULT_YGE15')]
print(h[['year', 'value', 'obs_status']].sort_values('year').round(2).to_string(index=False))
f = fact[(fact['indicator_code'] == 'UNE_DEAP_SEX_AGE_RT') & (fact['age_class'] == 'AGE_YTHADULT_YGE15') & (fact['year'].isin([2022, 2024]))]
print(f.pivot(index='sex', columns='year', values='value').round(2))
print('dim_indicator rows:', len(dim))
save_clean(dim, 'dim_indicator.csv'); save_clean(fact, 'fact_unemployment_ilo.csv')